# 04 — Test-set Evaluation

Final evaluation of the four ablation variants on the held-out **test split (n=2,573, never seen during training or validation)**. Produces the Chapter 6 primary results table plus four figures: per-variant confusion matrices, ROC overlay, F1 bar chart, and per-class precision/recall.

**Inputs (auto-discovered):**
- `configs/base.yaml` — model architecture (CLIP ViT-B/16 backbone, `fusion.dropout=0.2`).
- `cfg.checkpointing.dir` on Drive — one `*_best.pt` per variant. For `hemt_clip` the discovery prefers the *canonical* v4 seed=42 ckpt (filenames without `_seed*` suffix), not the seed-robustness runs.
- HDF5 from notebook 01 (17,149 rows; 2,573-row test split filtered via `f['splits']`).

**Outputs (under `outputs/eval/`, all paths relative to repo root):**

| File | Purpose |
|---|---|
| `summary_test.csv` / `.md` | 4-row Chapter 6 results table (val F1 + 5 test metrics + ckpt name) |
| `metrics_{variant}.json` × 4 | Full per-variant metrics (accuracy, F1, precision, recall, AUC-ROC, confusion matrix, per-class report) |
| `preds_{variant}.npz` × 4 | logits + probs + preds + labels — feeds notebook 05's XAI |
| `cm_{variant}.png` × 4 | Per-variant confusion matrix |
| `roc_overlay_test.png` | All 4 ROC curves on one axes |
| `f1_bar_test.png` | Ablation F1 bar chart |
| `per_class_pr_test.png` | Per-class precision/recall grouped bars |

**Runtime: ~3–4 min on T4** (inference only — no training, no backprop).

Run on Colab with the same Drive mount as notebooks 02/03 so checkpoints are accessible.

## Setup
One bootstrap cell — same idempotent pattern as notebooks 02 and 03. Mount Drive (pick **Account B** in the OAuth popup), clone-or-pull the repo, install deps, remove `jax`/`flax` (they force `numpy>=2`), copy the HDF5 from Drive to local SSD for faster sequential reads during inference.

In [ ]:
# Bootstrap — idempotent. Safe to re-run on a fresh or warm Colab runtime.
import os, sys, subprocess, shutil

os.environ['USE_FLAX'] = 'FALSE'
os.environ['USE_TF'] = 'FALSE'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
REPO_URL = 'https://github.com/staharizvi/hemt-clip-fnd.git'
REPO_DIR = '/content/hemt-clip-fnd'
H5_DRIVE = '/content/drive/MyDrive/hemt-clip-fnd/data/fakeddit.h5'
H5_LOCAL = '/content/fakeddit.h5'

if IN_COLAB:
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')

    if os.path.exists(os.path.join(REPO_DIR, '.git')):
        print('Repo present — pulling latest…')
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    else:
        print('Cloning repo…')
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)

    subprocess.run(['pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
    subprocess.run(['pip', 'uninstall', '-y', '-q', 'jax', 'jaxlib', 'flax'], check=False)

    if not os.path.exists(H5_LOCAL):
        if os.path.exists(H5_DRIVE):
            print(f'Copying {H5_DRIVE} -> {H5_LOCAL}…')
            shutil.copy(H5_DRIVE, H5_LOCAL)
        else:
            print(f'WARNING: {H5_DRIVE} not found.')
    else:
        print(f'h5 already at {H5_LOCAL}.')

    os.chdir(REPO_DIR)

print('\ncwd:', os.getcwd())
print('h5 :', H5_LOCAL, 'exists:', os.path.exists(H5_LOCAL))

## Point evaluator at the local HDF5
`evaluate.py` reads `cfg.data.hdf5_path` from `configs/base.yaml`. On Colab we want the local SSD copy (faster sequential reads) rather than the Drive path. Same patch pattern as notebook 03.

In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
if cfg['data']['hdf5_path'] != '/content/fakeddit.h5':
    cfg['data']['hdf5_path'] = '/content/fakeddit.h5'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Patched cfg.data.hdf5_path -> /content/fakeddit.h5')
else:
    print('cfg already points at local HDF5.')
print('checkpoints :', cfg['checkpointing']['dir'])

## Discover checkpoints

`training.evaluate.discover_checkpoints` matches `hemt_{variant}_*_best.pt` in the checkpoint dir, then for each variant picks the **most recently modified non-`_seed*` file**. This is deliberate:

- `text_only`: the v2 (B/32) ckpt is the latest — the text branch is backbone-independent, so it's still valid under the B/16 config.
- `image_only`, `concat_fusion`: latest = v4 (B/16) runs, the only ones we want.
- `hemt_clip`: has three v4 runs (seeds 42, 7, 123). The seed=42 run has no `_seed*` suffix and is the canonical headline ckpt (val F1=0.8229, top of the seed band); seeds 7 and 123 are robustness data, not headline numbers.

Override with `--checkpoints <path-to-json>` if needed.

In [ ]:
import sys
from pathlib import Path

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from training.evaluate import discover_checkpoints, VARIANTS

ckpt_dir = Path(cfg['checkpointing']['dir'])
print(f'checkpoint dir: {ckpt_dir}\n')

discovered = discover_checkpoints(ckpt_dir)
for variant in VARIANTS:
    ckpt = discovered[variant]
    size_mb = ckpt.stat().st_size / 1e6
    print(f'  {variant:<14s} {ckpt.name}  ({size_mb:.0f} MB)')

## Run test-set evaluation

Runs each variant sequentially on the test split (n=2,573). For each variant:
1. Build the model from `configs/base.yaml` (B/16 backbone).
2. Load the discovered `*_best.pt` weights into it.
3. Run inference in fp16 with `torch.no_grad()`.
4. Compute accuracy, F1 (binary + macro), precision, recall, AUC-ROC, confusion matrix.
5. Save `metrics_{variant}.json`, `preds_{variant}.npz`, `cm_{variant}.png`.

After the per-variant loop, builds the cross-variant artefacts: ROC overlay, F1 bar, per-class P/R, and the 4-row comparison table.

Wall-clock: **~3–4 min on T4** (~50s per variant).

In [ ]:
!python -m training.evaluate --split test

## Results — Chapter 6 comparison table

Columns: `val_f1` is the F1 baked into the checkpoint at save time (from notebook 03's training runs); `test_*` are the held-out numbers we just computed. A small val→test drop (~1 pt) is healthy; a large gap (>3 pt) would suggest val-set overfit.

In [ ]:
import pandas as pd

summary = pd.read_csv('outputs/eval/summary_test.csv').set_index('variant')
print(summary.to_string())
print()
print('val→test deltas (test_f1 − val_f1):')
for variant, row in summary.iterrows():
    print(f'  {variant:<14s}  {row["test_f1"] - row["val_f1"]:+.4f}')

## Figures

All PNGs are committed to `outputs/eval/` and drop directly into Chapter 6.

In [ ]:
from IPython.display import Image, display, Markdown

fig_dir = Path('outputs/eval')
for name, title in [
    ('f1_bar_test',       '### Ablation F1 bar — test split'),
    ('roc_overlay_test',  '### ROC overlay — test split'),
    ('per_class_pr_test', '### Per-class precision/recall'),
]:
    display(Markdown(title))
    display(Image(filename=str(fig_dir / f'{name}.png')))

display(Markdown('### Confusion matrices (per variant)'))
for variant in VARIANTS:
    display(Markdown(f'**{variant}**'))
    display(Image(filename=str(fig_dir / f'cm_{variant}.png')))

## Take-aways for Chapter 6 (fill in after the run)

Drop these bullets in once the cells above produce numbers — the structure should hold; the values get pasted in.

- **Test F1 ordering:** expected `text_only < image_only < concat_fusion ≤ hemt_clip` (matches val ordering, validates the ablation story end-to-end).
- **val→test delta** per variant — small drops (≤1 pt) confirm the 70/15/15 splits generalise; large drops would flag val-set overfit.
- **AUC-ROC** vs F1: AUC-ROC is threshold-agnostic; if hemt_clip's AUC lead is bigger than its F1 lead, the model has better-calibrated probabilities and a threshold sweep on val could squeeze more F1 (cheap follow-up).
- **Per-class P/R**: which class is harder for each variant? If `fake` recall consistently lags precision, the model is conservative on the positive class — worth flagging in Chapter 6's error analysis.
- **Confusion matrices**: dominant error mode for hemt_clip — is it false-positive (calling real news fake) or false-negative (missing actual fakes)? Has implications for deployment (a fact-checker tool that flags real news is more harmful than one that misses some fakes; a content-moderation tool is the inverse).

Once these bullets are concrete, proceed to notebook 05 (XAI artefacts) using the `preds_{variant}.npz` files saved above.